# Chapter 1 — AI Architecture & Agentic Foundations

## Objective

Build a minimal but complete security agent — detectors, a cognitive loop, and an audit log — and see where an agent's tools, and its attack surface, come from.

By the end you will be able to:

- Explain what an AI agent is and how a perceive → interpret → reason → act → learn loop works
- Build unsupervised, supervised, embedding-based, and generative detectors as agent tools
- Implement a cognitive loop that triages an alert and logs every tool call
- Describe least privilege and why an action log is the basis of Zero Trust
- Expose a tool to an agent through the Model Context Protocol

## Summary

An AI agent is a language model placed inside a loop: it perceives its environment, interprets what it sees, reasons toward a goal, acts through tools, and learns from the result. That autonomy is what makes agents useful for security operations — and what makes them dangerous, because every tool call is a decision the system makes on its own. In this notebook you build a minimal but complete security agent. You start with the machine-learning detectors that supply an agent's tools — anomaly detection, classification, embeddings, and a generative scorer — then assemble the cognitive loop that triages an alert, calls a read-only lookup tool, and records every action to an audit log. That log is the artifact everything else depends on.

## Outline

**Chapter arc:** from AI to agentic AI → agentic AI systems → the AI security ecosystem

This notebook covers:

- Unsupervised anomaly detection (Isolation Forest)
- Supervised detection (malware classifier)
- NLP detection (TF-IDF phishing)
- Embeddings and similarity (typosquat detection)
- Generative anomaly scoring (autoencoder)
- The cognitive loop
- Tool use and function calling
- Serving tools with the Model Context Protocol
- Runnable demos plus a live language-model example
- A self-check quiz

## The big picture

Modern AI is a stack of nested capabilities — machine learning inside deep learning inside generative AI — and an **agent** wraps a language model in an autonomy loop so it can perceive, reason, and act through tools. A few ideas frame everything that follows:

- **The autonomy ladder.** Chatbot → assistant → copilot → agent → multi-agent. Every rung adds capability *and* attack surface; prefer the least-autonomous design that meets the goal.
- **The cognitive loop.** An agent repeats *perceive → interpret → reason → act → learn* — a descendant of Boyd's OODA loop. Each phase is an injection point, which is why the loop logs every action.
- **Five attack-surface layers:** compute & storage, training data, training pipeline, the deployed model, and the **agent runtime** (tools, memory, orchestration) — the newest and least-defended layer.
- **Four frameworks name the threats:** the OWASP ML Top 10, the OWASP Top 10 for LLM Applications, NIST AI 100-2 (adversarial-ML taxonomy), and MITRE ATLAS.

**The cognitive loop**

<svg viewBox="0 0 984 154" width="100%" style="max-width:984px;height:auto;font-family:system-ui,-apple-system,Segoe UI,Arial,sans-serif" xmlns="http://www.w3.org/2000/svg" role="img"><defs><marker id="a" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#64748b"/></marker></defs><rect width="984" height="154" rx="12" fill="#ffffff"/><rect x="22" y="22" width="156" height="62" rx="9" fill="#1d4ed8"/><text x="100" y="53" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Perceive</text><line x1="178" y1="53" x2="218" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="218" y="22" width="156" height="62" rx="9" fill="#0f766e"/><text x="296" y="53" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Interpret</text><line x1="374" y1="53" x2="414" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="414" y="22" width="156" height="62" rx="9" fill="#b91c1c"/><text x="492" y="53" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Reason</text><line x1="570" y1="53" x2="610" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="610" y="22" width="156" height="62" rx="9" fill="#7c3aed"/><text x="688" y="53" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Act</text><line x1="766" y1="53" x2="806" y2="53" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><rect x="806" y="22" width="156" height="62" rx="9" fill="#a16207"/><text x="884" y="53" fill="#fff" font-size="15" font-weight="600" text-anchor="middle" dominant-baseline="middle">Learn</text><path d="M884,84 L884,114 L100,114 L100,84" fill="none" stroke="#64748b" stroke-width="2" marker-end="url(#a)"/><text x="492" y="107" fill="#64748b" font-size="11" text-anchor="middle">repeat</text><text x="492" y="136" fill="#334155" font-size="12.5" text-anchor="middle">Every phase is an injection point — poisoned perception, manipulated memory, hijacked action — which is why the loop logs every action.</text></svg>

## Setup

This notebook runs on **numpy**, **pandas**, and **scikit-learn**. The anomaly-detection demo reads the CICIDS-2017 sample `CICIDS-2017.csv` that ships in this folder (see **Data** below). Two examples are optional: the variational-autoencoder scorer uses **torch**, and the tool server uses **mcp** — both imported lazily, so every function *defines* with nothing installed. The core examples need no API key; one is wired below to enable an optional live language-model demo.


In [1]:
import os

# Load the OpenAI key from the key file into the environment (never printed).
KEY_PATH = '/home/student/keys/key.txt'
client = None
if os.path.exists(KEY_PATH):
    os.environ["OPENAI_API_KEY"] = open(KEY_PATH).read().strip()
    try:
        from openai import OpenAI
        client = OpenAI()
        print("OpenAI key loaded — client ready (key not shown).")
    except Exception as e:
        print("openai SDK unavailable; the live demo will be skipped:", e)
else:
    print(f"No key file at {KEY_PATH} — the live demo will be skipped.")

No key file at /home/student/keys/key.txt — the live demo will be skipped.


## Core building blocks

Each function below is introduced, then defined. Run them all, then see them in action in **Try it** and the **Live demo**.

### Unsupervised anomaly detection (Isolation Forest)

Isolation Forest needs **no labels**: it isolates outliers in fewer random splits, so a higher score means "more anomalous". With `contamination=` it flags ~1% of flows as outliers (`flag == -1`) — the rare traffic a SOC analyst should look at first.

> **Data:** the demo runs on a 100,000-flow sample of the **CICIDS-2017** intrusion-detection dataset, shipped with the course as `CICIDS-2017.csv` next to this notebook (rebuild it any time with `fetch_data.py` at the repo root). Full dataset (~2.8M labelled flows): [CIC-IDS2017 — Canadian Institute for Cybersecurity](https://www.unb.ca/cic/datasets/ids-2017.html) ([download mirror](https://huggingface.co/datasets/bencorn/CICIDS2017)).
> **Security lens:** anomaly detection is dual-use — the same model that finds intruders can be probed by an attacker; the action log is what lets a defender tell the two apart.


In [2]:
def isolation_forest_baseline(csv_path="CICIDS-2017.csv"):
    from sklearn.ensemble import IsolationForest
    import pandas as pd

    df = pd.read_csv(csv_path)
    X = df.drop(columns=["Label"])

    iso = IsolationForest(n_estimators=200, contamination=0.01,
                          random_state=42)
    iso.fit(X)                       # learn "normal"
    df["score"] = -iso.score_samples(X)   # higher = more anomalous
    df["flag"] = iso.predict(X)           # -1 outlier, 1 inlier
    return df


### Supervised detection (malware classifier)

These are the **labeled-data** counterparts: when you *have* ground truth, you classify rather than rank.

> **Security lens:** a trained detector is a *tool* an agent calls to enrich an alert — and, like any tool, a capability an attacker may try to evade or hijack.

In [3]:
def malware_classifier(features, labels):
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import train_test_split

    Xtr, Xte, ytr, yte = train_test_split(features, labels)
    clf = RandomForestClassifier(n_estimators=100)
    clf.fit(Xtr, ytr)                # learn benign vs malware
    print("accuracy:", clf.score(Xte, yte))
    return clf

### NLP detection (TF-IDF phishing)

These are the **labeled-data** counterparts: when you *have* ground truth, you classify rather than rank.

> **Security lens:** the same NLP that flags a lure also drafts convincing lures at scale — offensive understanding is a defensive necessity.

In [4]:
def phishing_detector(emails, labels):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression

    vec = TfidfVectorizer(max_features=5000)
    X = vec.fit_transform(emails)    # text -> vectors
    clf = LogisticRegression().fit(X, labels)
    pred = clf.predict(vec.transform(["Verify your account now"]))
    return vec, clf, pred

### Embeddings and similarity (typosquat detection)

An embedding maps a token (here, a domain name) to a vector; **semantic similarity becomes geometric closeness**. Cosine similarity reads `1.0` for "same direction/meaning" down to `~0` for unrelated.

> **Security lens:** the same vector geometry powers **memory poisoning** — an attacker plants text that embeds near legitimate queries so it is retrieved and trusted turns later.

In [5]:
def cosine(a, b):
    import numpy as np
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

# usage: cosine(emb["paypal.com"], emb["paypa1-secure.com"])  # ~0.93

### Generative anomaly scoring (autoencoder)

A variational autoencoder learns to reconstruct *normal* traffic; traffic it reconstructs poorly yields a high **reconstruction error**, which doubles as a per-sample **anomaly score** — a generative alternative to 's Isolation Forest.

> **Security lens:** two independent detectors agreeing on the same rare flows is stronger evidence than either alone.

In [6]:
def vae_anomaly_score(vae, x):
    import torch
    vae.eval()
    with torch.no_grad():
        x_hat, mu, logvar = vae(x)
    # per-sample reconstruction error = anomaly score
    return ((x - x_hat) ** 2).mean(dim=1)

# scores = vae_anomaly_score(vae, X_test)
# alerts = X_test[scores > threshold]   # flag outliers

### The cognitive loop

This is the heart of this lab. The loop runs **perceive → interpret → reason → act → learn**, and calls `log(action, result)` on **every** tool call.

> **Security lens:** the loop descends from Boyd's OODA loop; every phase is an injection point — poisoned perception, manipulated memory, hijacked action — so it logs every action.

In [7]:
def cognitive_loop(alert, goal, perceive, interpret, llm, tools,
                   memory, log, done):
    while not done(goal):
        obs = perceive(alert)             # 1. Perceive
        state = interpret(obs)            # 2. Interpret
        action = llm.reason(state, goal)  # 3. Reason
        result = tools.run(action)        # 4. Act
        memory.append(action, result)     # 5. Learn
        log(action, result)               # Zero Trust action log
        alert = result
    return memory.summary()

### Tool use and function calling

Before an agent can act, each capability is declared as a JSON schema: a name, a description, and typed parameters. The model emits a structured call; the agent executes it.

> **Security lens:** every registered tool is a capability an attacker can hijack. A `run_shell` added 'for convenience' plus an indirect injection becomes **excessive agency** (OWASP LLM06).

In [8]:
TOOLS = [{
    "name": "lookup_cve",
    "description": "Get CVSS score and summary for a CVE id",
    "parameters": {
        "type": "object",
        "properties": {
            "cve_id": {"type": "string",
                       "description": "e.g. CVE-2024-3094"},
        },
        "required": ["cve_id"],
    },
}]
# The model returns:
#   {"name": "lookup_cve", "arguments": {"cve_id": "CVE-2024-3094"}}

### Serving tools with the Model Context Protocol

The **Model Context Protocol** (<https://modelcontextprotocol.io>) is the open standard for serving tools to agents: *Host → Client → Server*, where the **tool schema is the authorization boundary** and the transport sets the perimeter. Here you wrap the anomaly detector as a single MCP tool, `query_anomaly(flow_id)`, over local `stdio`.

> **Security lens:** the tool schema — not the transport — is the authorization boundary: it defines exactly what a connecting host may call.

In [9]:
def build_mcp_server(detector):
    from mcp.server.fastmcp import FastMCP

    mcp = FastMCP("secops-tools")

    @mcp.tool()
    def query_anomaly(flow_id: str) -> dict:
        """Return the anomaly score for a network flow."""
        score = detector.score(flow_id)
        return {"flow_id": flow_id, "score": score}

    mcp.run(transport="stdio")   # local, low-latency

## Try it

Run the demos and read the output. The first cell needs no external data; the rest run the Isolation Forest baseline on the real CICIDS-2017 sample `CICIDS-2017.csv` shipped in this folder and keep its top-scored flow as the alert the agent demos below triage — if the file is missing they print how to rebuild it and move on.


In [10]:
# Smoke-test the label-free examples that need no external data.
import numpy as np
print("cosine identical:", round(cosine(np.array([1.0, 0]),
                                        np.array([1.0, 0])), 3))
print("function-calling schema tool:", TOOLS[0]["name"])
print("OK - see each function's docstring.")

cosine identical: 1.0
function-calling schema tool: lookup_cve
OK - see each function's docstring.


In [11]:
def run_isolation_forest_demo():
    df = isolation_forest_baseline("CICIDS-2017.csv")

    print("Inlier / Outlier counts:")
    print(df["flag"].map({1: "inlier", -1: "outlier"}).value_counts())

    print("\nTrue labels of the outliers:")
    print(df.loc[df["flag"] == -1, "Label"].value_counts())
    # Mostly unusual-but-benign flows, plus the truly rare attacks —
    # anomaly is not attack, which is why a SOC triages every flag.

    return df


In [12]:
# Run the detector on the real sample and keep its top-scored flow as
# the alert the agent demos below triage — a real score, not a fixed one.
import os
if os.path.exists("CICIDS-2017.csv"):
    df = run_isolation_forest_demo()
    top = df.loc[df["score"].idxmax()]
    alert = {"flow_id": f"flow-{top.name}",
             "score": round(float(top["score"]), 3),
             "cve": "CVE-2024-3094"}
    print("\nTop-scored flow:", alert, f"(true label: {top['Label']})")
else:
    print("No data file — run `python3 fetch_data.py` from the repo root first (skipped).")


Inlier / Outlier counts:
flag
inlier     101969
outlier      1030
Name: count, dtype: int64

True labels of the outliers:
Label
BENIGN           918
DoS slowloris     79
Infiltration      13
Heartbleed        11
DoS Hulk           4
Bot                4
PortScan           1
Name: count, dtype: int64

Top-scored flow: {'flow_id': 'flow-11389', 'score': 0.736, 'cve': 'CVE-2024-3094'} (true label: BENIGN)


## Live demo — a real reasoner inside the cognitive loop

The loop's **Reason** step calls `llm.reason(state, goal)`. Here we plug a real language model into that step and run one perceive → interpret → reason → act → learn cycle, recording the chosen action to the audit log. The alert it triages is real — the detector's top-scored flow from the **Try it** run above. Requires the API key wired above.


In [13]:
class LLMReasoner:
    def __init__(self, client, model="gpt-4o-mini"):
        self.client, self.model = client, model
    def reason(self, state, goal):
        r = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": "You are a SOC triage agent. Reply with ONE short next action (max 8 words)."},
                {"role": "user", "content": f"State: {state}\nGoal: {goal}\nNext action:"},
            ],
            temperature=0, max_tokens=24)
        return r.choices[0].message.content.strip()


class Memory:
    def __init__(self): self.items = []
    def append(self, action, result): self.items.append((action, result))
    def summary(self): return self.items


class Tools:
    def run(self, action): return f"executed: {action}"


if client and "alert" in dir():
    mem, trace = Memory(), []
    cognitive_loop(
        alert=alert,                      # the detector's real top-scored flow
        goal="triage the network anomaly",
        perceive=lambda a: a,
        interpret=lambda o: f"anomaly score {o['score']} on {o['flow_id']}",
        llm=LLMReasoner(client), tools=Tools(), memory=mem,
        log=lambda a, r: trace.append((a, r)),      # the action log
        done=lambda goal: len(mem.items) >= 1)
    print("LLM-chosen action :", mem.items[0][0])
    print("action-log entry  :", trace[0])
elif not client:
    print("skipped (no API key)")
else:
    print("skipped (run the Try-it data cell first)")


skipped (no API key)


### Worked flow: detector → alert → CVE lookup → triage

The alert is real — the top-scored flow the Isolation Forest found in the sample, so `score` comes from data, not a constant. The cognitive loop wires the read-only `lookup_cve` tool into the reasoning step: it enriches the alert, then escalates or monitors on the CVSS — logging every tool call to the action log. This runs deterministically, no API key needed.


In [14]:
# alert is REAL — produced by the detector in the Try-it cell above
# (recomputed quietly here if those cells were skipped).
import os
if "alert" not in dir():
    if os.path.exists("CICIDS-2017.csv"):
        df = isolation_forest_baseline()
        top = df.loc[df["score"].idxmax()]
        alert = {"flow_id": f"flow-{top.name}",
                 "score": round(float(top["score"]), 3),
                 "cve": "CVE-2024-3094"}
    else:
        alert = None

if not alert:
    print("No alert to triage — run the Try-it data cell above "
          "(and fetch_data.py if the data file is missing).")
else:
    print("alert from the detector:", alert)

    class _Mem:
        def __init__(self): self.entries = []
        def append(self, a, r): self.entries.append((a, r))
        def summary(self): return {"verdict": self.entries[-1][1], "tool_calls": len(self.entries)}

    class _Reasoner:                       # deterministic: enrich first, then decide
        def reason(self, state, goal):
            if state.get("cvss") is None:
                return {"name": "lookup_cve", "arguments": {"cve_id": state["cve"]}}
            verdict = "escalate" if state["cvss"] >= 7.0 else "monitor"
            return {"name": "triage", "arguments": {"verdict": verdict, "cve": state["cve"]}}

    class _CveTools:                       # least-privilege, read-only
        CVE_DB = {"CVE-2024-3094": {"cvss": 10.0, "severity": "CRITICAL"}}
        def run(self, action):
            n, a = action["name"], action["arguments"]
            if n == "lookup_cve":
                return {"cve": a["cve_id"], **self.CVE_DB.get(a["cve_id"], {"cvss": 0.0})}
            if n == "triage":
                return {"final_verdict": a["verdict"], "cve": a["cve"]}
            return {}

    _mem = _Mem()
    def _zt_log(a, r): print(f"[ZT-LOG] tool={a['name']:<10} args={a['arguments']} -> {r}")
    verdict = cognitive_loop(
        alert=alert,
        goal="triage: escalate or monitor",
        perceive=lambda al: al,
        interpret=lambda o: {"cve": o.get("cve"), "cvss": o.get("cvss")},
        llm=_Reasoner(), tools=_CveTools(), memory=_mem, log=_zt_log,
        done=lambda g: any(a["name"] == "triage" for a, _ in _mem.entries))
    print("AGENT VERDICT:", verdict)


alert from the detector: {'flow_id': 'flow-11389', 'score': 0.736, 'cve': 'CVE-2024-3094'}
[ZT-LOG] tool=lookup_cve args={'cve_id': 'CVE-2024-3094'} -> {'cve': 'CVE-2024-3094', 'cvss': 10.0, 'severity': 'CRITICAL'}
[ZT-LOG] tool=triage     args={'verdict': 'escalate', 'cve': 'CVE-2024-3094'} -> {'final_verdict': 'escalate', 'cve': 'CVE-2024-3094'}
AGENT VERDICT: {'verdict': {'final_verdict': 'escalate', 'cve': 'CVE-2024-3094'}, 'tool_calls': 2}


## Quiz

Answer each question, then fill in the `answers` dict in the next cell (`"A"`, `"B"`, …) and run it to grade yourself.

**1. With contamination=0.01, isolation_forest_baseline flags about 1% of flows. What does flag == -1 mean?**

- A. An inlier — normal traffic
- B. An outlier — anomalous traffic
- C. A labeling error
- D. A dropped row

**2. Why is calling log(action, result) on every tool call the seed of Zero Trust?**

- A. It speeds up the loop
- B. It makes every agent action auditable — never trust, always verify
- C. It encrypts the tools
- D. It lowers token usage

**3. The agent is given a single read-only lookup tool. Which principle does that illustrate?**

- A. Fail-open
- B. Least privilege — a small, auditable action surface
- C. Rate limiting
- D. Defense in depth

**4. cosine(a, b) is near 1.0 for two domain embeddings. What does that indicate?**

- A. They are unrelated
- B. The vectors point almost the same way — high similarity (possible typosquat)
- C. One vector is zero
- D. An error

**5. In the Model Context Protocol, what is the authorization boundary?**

- A. The network port
- B. The tool schema — it defines what an agent may call
- C. The model weights
- D. The prompt


In [15]:
import base64
_KEY = {'q1': 'Qg==', 'q2': 'Qg==', 'q3': 'Qg==', 'q4': 'Qg==', 'q5': 'Qg=='}   # base64 so the answers aren't obvious at a glance
_WHY = {'q1': '-1 is an outlier; a higher score means more anomalous.', 'q2': 'Zero Trust requires every action to be verifiable; the log is that evidence.', 'q3': 'Fewer, narrower tools mean a smaller blast radius if the agent is hijacked.', 'q4': 'Cosine ~1 means near-identical direction/meaning.', 'q5': 'The schema is what a host is allowed to call; it gates access.'}

def check(answers):
    key = {q: base64.b64decode(v).decode() for q, v in _KEY.items()}
    if all(str(answers.get(q, '')).strip().upper() in ('', '?') for q in key):
        print('Fill in the answers dict above (A/B/C/D) and re-run to grade.')
        return
    score = 0
    for q in sorted(key):
        got = str(answers.get(q, '')).strip().upper()
        if got == key[q]:
            score += 1; print(f'\u2713 {q}: {got}')
        else:
            print(f'\u2717 {q}: you put {got or "\u2014"} \u2014 correct is {key[q]}. {_WHY[q]}')
    print(f'\nScore: {score}/{len(key)}')

In [16]:
answers = {
    "q1": "?",   # A / B / C / D
    "q2": "?",   # A / B / C / D
    "q3": "?",   # A / B / C / D
    "q4": "?",   # A / B / C / D
    "q5": "?",   # A / B / C / D
}
check(answers)

Fill in the answers dict above (A/B/C/D) and re-run to grade.


## Key takeaways

- An agent is a perceive → interpret → reason → act → learn loop over tools; autonomy means every tool call is a decision to secure.
- Detectors (isolation forest, classifiers, embeddings, autoencoders) are the *tools* an agent calls, not the agent itself.
- Least privilege — one read-only tool — keeps the action surface small and auditable.
- Logging every action is the foundation every later governance control builds on.